# v8 Residual LightGBM Optimized from Public Notebooks

Conservative v8 candidate based on v3 residual LightGBM. It preserves the residual-anchor framework, adds compact public-inspired features, two-seed LightGBM residual ensembling, mild residual clipping, gated blending, and lightweight target-free path/curve correction. Leakage guard comments are included where offline feature groups use full hidden covariates but never hidden `TVT` labels.

README update completed outside this notebook; keep v3 selected unless this v8 notebook beats the v3 Public LB RMSE 14.527 on Kaggle without memory or leakage concerns.


## Config


In [ ]:
from __future__ import annotations

import gc
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 20260507
SEEDS = [42, 2024]
RUN_LOCAL_VALIDATION = False
VALIDATION_FRAC_WELLS = 0.15
OUTPUT_PATH = Path("/kaggle/working/submission.csv")
KAGGLE_INPUT_ROOT = Path("/kaggle/input")
PREFERRED_INPUT_DIRS = [
    KAGGLE_INPUT_ROOT / "rogii-wellbore-geology-prediction",
    KAGGLE_INPUT_ROOT / "competitions" / "rogii-wellbore-geology-prediction",
]
TARGET = "TVT"
SUBMISSION_TARGET = "tvt"

# v8 conservative optimization switches. All features remain target-free at inference.
USE_GATED_BLEND = True
USE_CURVE_CORRECTION = True
USE_RESIDUAL_CLIPPING = True
CURVE_CORRECTION_WEIGHT = 0.06
RESIDUAL_CLIP_QUANTILES = (0.003, 0.997)
MEMORY_DROP_RSS_MB = 24_000.0

OFFSETS = [-60, -40, -25, -15, -8, 0, 8, 15, 25, 40, 60]
LOCAL_OFFSET_GRID = [-80, -60, -40, -25, -15, -8, 0, 8, 15, 25, 40, 60, 80]
PATH_ENDPOINT_GRID = [-100, -80, -60, -45, -30, -20, -12, 0, 12, 20, 30, 45, 60, 80, 100]
RECENT_SLOPE_WINDOWS = [20, 50, 100, 200]
OPTIONAL_FEATURE_PREFIXES = ("off_path_absdiff_", "off_path_geo_code_")
HORIZONTAL_COLS = ["MD", "X", "Y", "Z", "GR", "TVT_input"]
TRAIN_HORIZONTAL_COLS = HORIZONTAL_COLS + [TARGET]
TYPEWELL_COLS = ["TVT", "GR", "Geology"]

LGB_PARAMS = dict(
    objective="regression",
    metric="rmse",
    n_estimators=1700,
    learning_rate=0.028,
    num_leaves=112,
    min_child_samples=90,
    subsample=0.85,
    subsample_freq=1,
    colsample_bytree=0.78,
    reg_alpha=0.05,
    reg_lambda=1.25,
    random_state=SEED,
    n_jobs=-1,
    verbose=-1,
    force_col_wise=True,
)

random.seed(SEED)
np.random.seed(SEED)
print("v8 switches:")
print("USE_GATED_BLEND=", USE_GATED_BLEND)
print("USE_CURVE_CORRECTION=", USE_CURVE_CORRECTION)
print("USE_RESIDUAL_CLIPPING=", USE_RESIDUAL_CLIPPING)
print("SEEDS=", SEEDS)
print("RUN_LOCAL_VALIDATION=", RUN_LOCAL_VALIDATION)


## Imports and utilities

In [ ]:
def process_memory_mb() -> float:
    try:
        import psutil
        return float(psutil.Process(os.getpid()).memory_info().rss / 1024**2)
    except Exception:
        try:
            import resource
            return float(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024.0)
        except Exception:
            return float("nan")


def frame_memory_mb(df: pd.DataFrame | None) -> float:
    if df is None:
        return 0.0
    return float(df.memory_usage(deep=True).sum() / 1024**2)


def log_memory(stage: str, **frames: pd.DataFrame) -> None:
    details = ", ".join(f"{name}={frame_memory_mb(df):.1f}MB" for name, df in frames.items())
    suffix = f" ({details})" if details else ""
    print(f"[memory] {stage}: rss={process_memory_mb():.1f}MB{suffix}")


def find_input_dir() -> Path:
    candidates = []
    for path in PREFERRED_INPUT_DIRS:
        if (path / "sample_submission.csv").exists() and (path / "train").is_dir() and (path / "test").is_dir():
            candidates.append(path)
    if KAGGLE_INPUT_ROOT.exists():
        for sample_path in sorted(KAGGLE_INPUT_ROOT.rglob("sample_submission.csv")):
            root = sample_path.parent
            if (root / "train").is_dir() and (root / "test").is_dir():
                candidates.append(root)
    unique = []
    seen = set()
    for path in candidates:
        key = str(path)
        if key not in seen:
            seen.add(key)
            unique.append(path)
    if not unique:
        available = sorted(str(p) for p in KAGGLE_INPUT_ROOT.glob("*")) if KAGGLE_INPUT_ROOT.exists() else []
        raise FileNotFoundError(f"Could not find competition data under /kaggle/input. Available: {available[:20]}")
    return unique[0]


def well_id_from_path(path: Path) -> str:
    return path.name.split("__", 1)[0].split(".", 1)[0]


def existing_usecols(path: Path, requested: list[str]) -> list[str]:
    header = pd.read_csv(path, nrows=0).columns.tolist()
    return [col for col in requested if col in header]


def read_csv_small(path: Path, requested: list[str]) -> pd.DataFrame:
    df = pd.read_csv(path, usecols=existing_usecols(path, requested), low_memory=True)
    for col in df.columns:
        if col == "Geology":
            continue
        df[col] = pd.to_numeric(df[col], errors="coerce", downcast="float")
    return df


def finite_stat(values, kind: str) -> float:
    arr = np.asarray(values, dtype="float64")
    arr = arr[np.isfinite(arr)]
    if len(arr) == 0:
        return np.nan
    if kind == "mean":
        return float(np.mean(arr))
    if kind == "std":
        return float(np.std(arr))
    if kind == "min":
        return float(np.min(arr))
    if kind == "max":
        return float(np.max(arr))
    if kind.startswith("p"):
        return float(np.quantile(arr, float(kind[1:]) / 100.0))
    raise ValueError(kind)


def finite_slope(x, y) -> float:
    x = np.asarray(x, dtype="float64")
    y = np.asarray(y, dtype="float64")
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 3 or np.nanstd(x[mask]) < 1e-9:
        return 0.0
    return float(np.polyfit(x[mask], y[mask], 1)[0])


def interp_sorted(x, y, target):
    x = np.asarray(x, dtype="float64")
    y = np.asarray(y, dtype="float64")
    target = np.asarray(target, dtype="float64")
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 2:
        return np.full(len(target), np.nan, dtype="float64")
    xs = x[mask]
    ys = y[mask]
    order = np.argsort(xs)
    xs = xs[order]
    ys = ys[order]
    xs, idx = np.unique(xs, return_index=True)
    ys = ys[idx]
    if len(xs) < 2:
        return np.full(len(target), np.nan, dtype="float64")
    return np.interp(target, xs, ys, left=np.nan, right=np.nan)


def gradient(values, x):
    values = np.asarray(values, dtype="float64")
    x = np.asarray(x, dtype="float64")
    if len(values) < 2:
        return np.zeros(len(values), dtype="float64")
    filled = pd.Series(values).interpolate(limit_direction="both").ffill().bfill().fillna(0.0).to_numpy(dtype="float64")
    x_filled = pd.Series(x).interpolate(limit_direction="both").ffill().bfill().fillna(0.0).to_numpy(dtype="float64")
    if np.nanstd(x_filled) < 1e-9:
        return np.zeros(len(values), dtype="float64")
    return np.gradient(filled, x_filled)


def rolling(values, window: int, method: str):
    s = pd.Series(values, dtype="float64")
    if method == "median":
        out = s.rolling(window, min_periods=1, center=True).median()
    elif method == "std":
        out = s.rolling(window, min_periods=2, center=True).std()
    else:
        out = s.rolling(window, min_periods=1, center=True).mean()
    return out.interpolate(limit_direction="both").ffill().bfill().fillna(0.0).to_numpy(dtype="float64")


def robust_scale_fit(x, y):
    x = np.asarray(x, dtype="float64")
    y = np.asarray(y, dtype="float64")
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 20 or np.nanstd(x[mask]) < 1e-6:
        return 1.0, 0.0
    xs = x[mask]
    ys = y[mask]
    qx = np.nanquantile(xs, [0.1, 0.9])
    qy = np.nanquantile(ys, [0.1, 0.9])
    denom = qx[1] - qx[0]
    a = 1.0 if abs(denom) < 1e-6 else (qy[1] - qy[0]) / denom
    b = np.nanmedian(ys) - a * np.nanmedian(xs)
    return float(a), float(b)


def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2)))

## Typewell helpers

In [ ]:
GEOLOGY_ORDER = ["ANCC", "ASTNU", "ASTNL", "EGFDU", "EGFDL", "LFHL", "LFGT", "LBHL", "MNSS", "BUDA"]
GEOLOGY_CODE = {name: i for i, name in enumerate(GEOLOGY_ORDER)}


def geology_arrays(typewell: pd.DataFrame):
    if "TVT" not in typewell.columns:
        return np.array([0.0]), np.array([-1], dtype="int16")
    tw = typewell.copy()
    tw = tw[pd.to_numeric(tw["TVT"], errors="coerce").notna()].sort_values("TVT")
    if tw.empty:
        return np.array([0.0]), np.array([-1], dtype="int16")
    tvt = tw["TVT"].to_numpy(dtype="float64")
    if "Geology" in tw.columns:
        labels = tw["Geology"].fillna("<NA>").astype(str).to_numpy()
        codes = np.array([GEOLOGY_CODE.get(x, -1) for x in labels], dtype="int16")
    else:
        codes = np.full(len(tw), -1, dtype="int16")
    return tvt, codes


def geology_code_at(tvt_grid, codes, target):
    target = np.asarray(target, dtype="float64")
    if len(tvt_grid) == 0 or len(codes) == 0:
        return np.full(len(target), -1, dtype="float64")
    idx = np.searchsorted(tvt_grid, target, side="right") - 1
    idx = np.clip(idx, 0, len(tvt_grid) - 1)
    return codes[idx].astype("float64")


def geology_boundary_features(tvt_grid, codes, last_tvt):
    if len(tvt_grid) < 2 or len(codes) < 2:
        return {
            "last_geo_code": -1.0,
            "last_geo_prev_boundary_dist": np.nan,
            "last_geo_next_boundary_dist": np.nan,
            "last_geo_interval_thickness": np.nan,
            "last_geo_boundary_balance": np.nan,
        }
    changes = np.flatnonzero(codes[1:] != codes[:-1]) + 1
    boundaries = tvt_grid[changes] if len(changes) else np.array([], dtype="float64")
    last_code = float(geology_code_at(tvt_grid, codes, np.array([last_tvt]))[0])
    prev_boundaries = boundaries[boundaries <= last_tvt]
    next_boundaries = boundaries[boundaries > last_tvt]
    prev_dist = float(last_tvt - prev_boundaries[-1]) if len(prev_boundaries) else np.nan
    next_dist = float(next_boundaries[0] - last_tvt) if len(next_boundaries) else np.nan
    thickness = prev_dist + next_dist if np.isfinite(prev_dist) and np.isfinite(next_dist) else np.nan
    balance = (next_dist - prev_dist) / (thickness + 1e-6) if np.isfinite(thickness) else np.nan
    return {
        "last_geo_code": last_code,
        "last_geo_prev_boundary_dist": prev_dist,
        "last_geo_next_boundary_dist": next_dist,
        "last_geo_interval_thickness": thickness,
        "last_geo_boundary_balance": balance,
    }

## Well-level feature builder

In [ ]:
def summarize_known_gap(h, gap_mask, prefix_mask, last_idx):
    out = {
        "n_rows": float(len(h)),
        "known_n": float(prefix_mask.sum()),
        "gap_n": float(gap_mask.sum()),
        "gap_frac": float(gap_mask.sum() / max(1, len(h))),
        "gap_start_idx": float(np.flatnonzero(gap_mask)[0]),
        "last_known_md": float(h.loc[last_idx, "MD"]),
        "last_known_x": float(h.loc[last_idx, "X"]),
        "last_known_y": float(h.loc[last_idx, "Y"]),
        "last_known_z": float(h.loc[last_idx, "Z"]),
    }
    for label, mask in [("known", prefix_mask), ("gap", gap_mask)]:
        for col in ["GR", "Z", "MD", "X", "Y"]:
            vals = h.loc[mask, col].to_numpy(dtype="float64") if col in h.columns else np.array([], dtype="float64")
            out[f"{label}_{col.lower()}_nan_frac"] = float(1.0 - np.isfinite(vals).mean()) if len(vals) else np.nan
            if col == "GR" or (label == "gap" and col in ["Z", "MD", "X", "Y"]):
                for stat in ["mean", "std", "min", "max"]:
                    out[f"{label}_{col.lower()}_{stat}"] = finite_stat(vals, stat)
    for window in RECENT_SLOPE_WINDOWS:
        idx = np.flatnonzero(prefix_mask)[-window:]
        for col in ["TVT_input", "GR", "Z", "X", "Y"]:
            out[f"last{window}_{col.lower()}_slope"] = finite_slope(h.loc[idx, "MD"].to_numpy(dtype="float64"), h.loc[idx, col].to_numpy(dtype="float64")) if len(idx) else 0.0
    gap_idx = np.flatnonzero(gap_mask)
    if len(gap_idx):
        out["gap_md_span"] = float(h.loc[gap_idx[-1], "MD"] - h.loc[gap_idx[0], "MD"])
        out["gap_z_delta"] = float(h.loc[gap_idx[-1], "Z"] - h.loc[gap_idx[0], "Z"])
        out["gap_x_delta"] = float(h.loc[gap_idx[-1], "X"] - h.loc[gap_idx[0], "X"])
        out["gap_y_delta"] = float(h.loc[gap_idx[-1], "Y"] - h.loc[gap_idx[0], "Y"])
        out["gap_xy_span"] = float(np.sqrt(out["gap_x_delta"] ** 2 + out["gap_y_delta"] ** 2))
        out["gap_z_over_xy"] = float(out["gap_z_delta"] / (out["gap_xy_span"] + 1e-6))
    return out


def best_two_gap(matrix: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    finite = np.where(np.isfinite(matrix), matrix, np.inf)
    best_idx = np.argmin(finite, axis=0).astype("float64")
    sorted_vals = np.sort(finite, axis=0)
    min_abs = sorted_vals[0]
    top2_gap = sorted_vals[1] - sorted_vals[0] if sorted_vals.shape[0] > 1 else np.zeros_like(min_abs)
    min_abs = np.where(np.isfinite(min_abs), min_abs, np.nan)
    top2_gap = np.where(np.isfinite(top2_gap), top2_gap, 0.0)
    return min_abs, best_idx, top2_gap


def make_well_feature_frame(h_path: Path, split: str) -> pd.DataFrame:
    well = well_id_from_path(h_path)
    h = read_csv_small(h_path, TRAIN_HORIZONTAL_COLS if split == "train" else HORIZONTAL_COLS)
    t_path = h_path.with_name(f"{well}__typewell.csv")
    t = read_csv_small(t_path, TYPEWELL_COLS) if t_path.exists() else pd.DataFrame(columns=TYPEWELL_COLS)
    for col in HORIZONTAL_COLS:
        if col not in h.columns:
            h[col] = np.nan
    gap = h["TVT_input"].isna().to_numpy()
    gap_idx = np.flatnonzero(gap)
    known = (~gap) & h["TVT_input"].notna().to_numpy()
    if len(gap_idx) == 0:
        return pd.DataFrame()
    known_before = np.flatnonzero(known & (np.arange(len(h)) < gap_idx[0]))
    if len(known_before) == 0:
        return pd.DataFrame()
    last_idx = int(known_before[-1])
    prefix = known & (np.arange(len(h)) <= last_idx)
    last_tvt = float(h.loc[last_idx, "TVT_input"])
    last_md = float(h.loc[last_idx, "MD"])
    gap_len = len(gap_idx)
    row_pos = np.arange(gap_len, dtype="float64")
    pos = row_pos / max(1, gap_len - 1)
    ease_pos = 3.0 * pos ** 2 - 2.0 * pos ** 3

    well_features = summarize_known_gap(h, gap, prefix, last_idx)
    tw_tvt = t["TVT"].to_numpy(dtype="float64") if "TVT" in t.columns else np.array([], dtype="float64")
    tw_gr = t["GR"].to_numpy(dtype="float64") if "GR" in t.columns else np.array([], dtype="float64")
    geo_tvt, geo_codes = geology_arrays(t)
    well_features.update(geology_boundary_features(geo_tvt, geo_codes, last_tvt))

    known_tvt = h.loc[prefix, "TVT_input"].to_numpy(dtype="float64")
    known_gr = h.loc[prefix, "GR"].to_numpy(dtype="float64")
    tw_known_gr = interp_sorted(tw_tvt, tw_gr, known_tvt)
    tw_scale_a, tw_scale_b = robust_scale_fit(tw_known_gr, known_gr)
    corr_mask = np.isfinite(tw_known_gr) & np.isfinite(known_gr)
    well_features["known_typewell_gr_corr"] = float(np.corrcoef(tw_known_gr[corr_mask], known_gr[corr_mask])[0, 1]) if corr_mask.sum() >= 20 and np.nanstd(tw_known_gr[corr_mask]) > 1e-6 and np.nanstd(known_gr[corr_mask]) > 1e-6 else 0.0
    well_features["typewell_gr_scale_a"] = tw_scale_a
    well_features["typewell_gr_scale_b"] = tw_scale_b
    near_last = np.abs(tw_tvt - last_tvt) <= 120 if len(tw_tvt) else np.array([], dtype=bool)
    well_features["typewell_gr_mean_near_last120"] = finite_stat(tw_gr[near_last], "mean") if len(tw_gr) else np.nan
    well_features["typewell_gr_std_near_last120"] = finite_stat(tw_gr[near_last], "std") if len(tw_gr) else np.nan
    well_features["typewell_gr_slope_near_last120"] = finite_slope(tw_tvt[near_last], tw_gr[near_last]) if len(tw_gr) else 0.0

    md = h.loc[gap, "MD"].to_numpy(dtype="float64")
    x = h.loc[gap, "X"].to_numpy(dtype="float64")
    y = h.loc[gap, "Y"].to_numpy(dtype="float64")
    z = h.loc[gap, "Z"].to_numpy(dtype="float64")
    gr = h.loc[gap, "GR"].to_numpy(dtype="float64")
    gr_roll17 = rolling(gr, 17, "median")
    gr_roll51 = rolling(gr, 51, "mean")
    gr_std51 = rolling(gr, 51, "std")
    gr_roll151 = rolling(gr, 151, "mean")
    gr_d1 = gradient(gr, md)
    gr_d2 = gradient(gr_d1, md)
    z_d1 = gradient(z, md)
    z_d2 = gradient(z_d1, md)
    x_d1 = gradient(x, md)
    y_d1 = gradient(y, md)
    azimuth = np.arctan2(y - y[0], x - x[0])
    xy_dist = np.sqrt((x - x[0]) ** 2 + (y - y[0]) ** 2)
    curvature = gradient(np.unwrap(azimuth), np.maximum(xy_dist, 1.0))
    known_gr_mean = well_features.get("known_gr_mean", 0.0)
    known_gr_std = well_features.get("known_gr_std", 0.0)
    gap_gr_mean = well_features.get("gap_gr_mean", 0.0)
    gap_gr_std = well_features.get("gap_gr_std", 0.0)
    gr_z_known = (gr - known_gr_mean) / (known_gr_std + 1e-6)
    gr_z_gap = (gr - gap_gr_mean) / (gap_gr_std + 1e-6)
    gr_filled = pd.Series(gr).interpolate(limit_direction="both").ffill().bfill().fillna(0.0).to_numpy(dtype="float64")
    gr_highfreq = gr_filled - gr_roll151
    gr_cum_anom = np.cumsum(np.nan_to_num(gr_z_gap, nan=0.0)) / np.sqrt(row_pos + 1.0)

    recent_slope = well_features.get("last100_tvt_input_slope", 0.0)
    if not np.isfinite(recent_slope):
        recent_slope = 0.0
    baseline_tvt = last_tvt + float(recent_slope) * (md - last_md)

    data = {
        "well": np.array([well] * gap_len),
        "row_index": gap_idx.astype("int32"),
        "id": np.array([f"{well}_{idx}" for idx in gap_idx]),
        "target_residual": (h.loc[gap, TARGET].to_numpy(dtype="float64") - last_tvt) if TARGET in h.columns else np.full(gap_len, np.nan),
        "last_known_tvt_row": np.full(gap_len, last_tvt),
        "tail_row_number": row_pos,
        "off_tail_frac": pos,
        "off_tail_frac2": pos ** 2,
        "off_tail_frac3": pos ** 3,
        "off_sin_tail_frac_pi": np.sin(np.pi * pos),
        "pos": pos,
        "pos_centered": pos - 0.5,
        "pos2": pos ** 2,
        "sqrt_pos": np.sqrt(pos),
        "log1p_row": np.log1p(row_pos),
        "md_delta": md - last_md,
        "md_delta_norm": (md - md[0]) / (md[-1] - md[0] + 1e-6),
        "z_delta_from_last": z - float(h.loc[last_idx, "Z"]),
        "z_delta_from_gap0": z - z[0],
        "x_delta_from_gap0": x - x[0],
        "y_delta_from_gap0": y - y[0],
        "xy_dist_from_gap0": xy_dist,
        "xy_dist_norm": xy_dist / (xy_dist[-1] + 1.0),
        "azimuth_sin": np.sin(azimuth),
        "azimuth_cos": np.cos(azimuth),
        "z_d1": z_d1,
        "z_d2": z_d2,
        "x_d1": x_d1,
        "y_d1": y_d1,
        "trajectory_curvature": curvature,
        "gr": gr,
        "gr_missing": (~np.isfinite(gr)).astype("float64"),
        "gr_roll17": gr_roll17,
        "gr_roll51": gr_roll51,
        "gr_std51": gr_std51,
        "gr_z_known": gr_z_known,
        "gr_z_gap": gr_z_gap,
        "gr_d1": gr_d1,
        "gr_d2": gr_d2,
        "gr_abs_d1": np.abs(gr_d1),
        "gr_highfreq": gr_highfreq,
        "gr_cum_anom": gr_cum_anom,
        "prefix_recent_tvt_slope_md_20": np.full(gap_len, well_features.get("last20_tvt_input_slope", 0.0)),
        "prefix_recent_tvt_slope_md_100": np.full(gap_len, well_features.get("last100_tvt_input_slope", 0.0)),
        "prefix_recent_tvt_slope_md_200": np.full(gap_len, well_features.get("last200_tvt_input_slope", 0.0)),
        "prefix_baseline_tvt": baseline_tvt,
        "prefix_baseline_delta": baseline_tvt - last_tvt,
    }

    tw_abs_candidates = []
    for offset in OFFSETS:
        candidate_tvt = last_tvt + float(offset) * pos
        candidate_gr = tw_scale_a * interp_sorted(tw_tvt, tw_gr, candidate_tvt) + tw_scale_b
        diff = gr_roll17 - candidate_gr
        suffix = f"offset_{offset:+d}".replace("+", "p").replace("-", "m")
        data[f"tw_gr_diff_{suffix}"] = diff
        data[f"tw_gr_absdiff_{suffix}"] = np.abs(diff)
        data[f"tw_geo_code_{suffix}"] = geology_code_at(geo_tvt, geo_codes, candidate_tvt)
        tw_abs_candidates.append(np.abs(diff))
    tw_abs = np.vstack(tw_abs_candidates) if tw_abs_candidates else np.full((1, gap_len), np.nan)
    data["tw_candidate_min_absdiff"], data["tw_candidate_best_index"], data["tw_candidate_top2_gap"] = best_two_gap(tw_abs)

    local_abs_candidates = []
    for offset in LOCAL_OFFSET_GRID:
        local_tvt = baseline_tvt + float(offset)
        local_gr = tw_scale_a * interp_sorted(tw_tvt, tw_gr, local_tvt) + tw_scale_b
        diff = gr_roll17 - local_gr
        suffix = f"{offset:+d}".replace("+", "p").replace("-", "m")
        data[f"tw_local_diff_{suffix}"] = diff
        data[f"tw_local_absdiff_{suffix}"] = np.abs(diff)
        local_abs_candidates.append(np.abs(diff))
    local_abs = np.vstack(local_abs_candidates) if local_abs_candidates else np.full((1, gap_len), np.nan)
    data["tw_local_min_absdiff"], local_best_idx, data["tw_local_top2_gap"] = best_two_gap(local_abs)
    data["tw_local_best_offset"] = np.asarray(LOCAL_OFFSET_GRID, dtype="float64")[np.clip(local_best_idx.astype(int), 0, len(LOCAL_OFFSET_GRID) - 1)]

    path_abs_candidates = []
    path_weights = []
    for endpoint in PATH_ENDPOINT_GRID:
        path_tvt = last_tvt + float(endpoint) * ease_pos
        path_gr = tw_scale_a * interp_sorted(tw_tvt, tw_gr, path_tvt) + tw_scale_b
        diff = gr_roll51 - path_gr
        suffix = f"{endpoint:+d}".replace("+", "p").replace("-", "m")
        absdiff = np.abs(diff)
        data[f"off_path_absdiff_{suffix}"] = absdiff
        data[f"off_path_geo_code_{suffix}"] = geology_code_at(geo_tvt, geo_codes, path_tvt)
        path_abs_candidates.append(absdiff)
        path_weights.append(np.exp(-np.clip(absdiff, 0, 100) / 8.0))
    path_abs = np.vstack(path_abs_candidates) if path_abs_candidates else np.full((1, gap_len), np.nan)
    path_weight_matrix = np.vstack(path_weights) if path_weights else np.zeros((1, gap_len))
    data["off_path_min_absdiff"], path_best_idx, data["off_path_top2_gap"] = best_two_gap(path_abs)
    endpoints = np.asarray(PATH_ENDPOINT_GRID, dtype="float64")
    data["off_path_best_endpoint"] = endpoints[np.clip(path_best_idx.astype(int), 0, len(endpoints) - 1)]
    weight_sum = np.maximum(np.nansum(path_weight_matrix, axis=0), 1e-9)
    data["off_path_soft_endpoint_mean"] = np.nansum(path_weight_matrix * endpoints[:, None], axis=0) / weight_sum
    data["off_path_prior_delta"] = data["off_path_soft_endpoint_mean"] * ease_pos

    # Leakage guard: all local/path features use target-free hidden GR, trajectory, and typewell reference data.
    # They never read hidden TVT labels, train-only surface columns, or sample-submission values.
    for key, value in well_features.items():
        data[key] = np.full(gap_len, value)
    out = pd.DataFrame(data)
    for col in out.columns:
        if col in ["well", "id"]:
            continue
        out[col] = pd.to_numeric(out[col], errors="coerce", downcast="float")
    return out


def build_feature_table(data_dir: Path, split: str) -> pd.DataFrame:
    frames = []
    paths = sorted((data_dir / split).glob("*__horizontal_well.csv"))
    for i, path in enumerate(paths, start=1):
        frame = make_well_feature_frame(path, split)
        if not frame.empty:
            frames.append(frame)
        if i % 100 == 0:
            print(f"built {split} wells: {i}/{len(paths)}")
    if not frames:
        raise RuntimeError(f"No feature rows built for {split}")
    result = pd.concat(frames, axis=0, ignore_index=True)
    del frames
    gc.collect()
    return result


## Build features and matrices


In [ ]:
def print_stats(label: str, values) -> None:
    arr = np.asarray(values, dtype="float64")
    arr = arr[np.isfinite(arr)]
    if len(arr) == 0:
        print(f"{label} min/max/mean/std: all non-finite")
        return
    print(f"{label} min/max/mean/std:", float(np.min(arr)), float(np.max(arr)), float(np.mean(arr)), float(np.std(arr)))


DATA_DIR = find_input_dir()
print(f"DATA_DIR={DATA_DIR}")
log_memory("after data discovery")

sample = pd.read_csv(DATA_DIR / "sample_submission.csv")
train_paths = sorted((DATA_DIR / "train").glob("*__horizontal_well.csv"))
test_paths = sorted((DATA_DIR / "test").glob("*__horizontal_well.csv"))
print("sample_submission rows:", len(sample))
print("number of train wells:", len(train_paths))
print("number of test wells:", len(test_paths))

train_df = build_feature_table(DATA_DIR, "train")
train_df = train_df[np.isfinite(train_df["target_residual"])].reset_index(drop=True)
log_memory("after train feature build", train_df=train_df)

resid = train_df["target_residual"].to_numpy(dtype="float64")
print_stats("last_known_tvt_row", train_df["last_known_tvt_row"])
print_stats("target_residual", resid)

test_df = build_feature_table(DATA_DIR, "test")
log_memory("after test feature build", test_df=test_df)
print("train rows:", len(train_df))
print("test rows:", len(test_df))
print("hidden gap rows:", len(test_df))

DROP_COLS = {"well", "id", "row_index", "target_residual"}
FEATURE_COLUMNS = [col for col in train_df.columns if col not in DROP_COLS and pd.api.types.is_numeric_dtype(train_df[col])]
missing_test = [col for col in FEATURE_COLUMNS if col not in test_df.columns]
if missing_test:
    raise RuntimeError(f"Missing test feature columns: {missing_test[:20]}")

optional_cols = [col for col in FEATURE_COLUMNS if col.startswith(OPTIONAL_FEATURE_PREFIXES)]
dropped_optional_groups = []
if process_memory_mb() > MEMORY_DROP_RSS_MB and optional_cols:
    FEATURE_COLUMNS = [col for col in FEATURE_COLUMNS if col not in optional_cols]
    dropped_optional_groups.append("candidate path optional features")

included_groups = []
for prefix, label in [
    ("off_tail_", "tail/gap position"),
    ("prefix_recent_", "prefix recent slopes"),
    ("tw_local_", "local typewell alignment"),
    ("off_path_", "lightweight candidate-path prior"),
    ("last_geo_", "typewell geology boundary context"),
]:
    if any(col.startswith(prefix) for col in FEATURE_COLUMNS):
        included_groups.append(label)
print("feature count:", len(FEATURE_COLUMNS))
print("top feature groups included:", included_groups)
print("optional feature groups dropped:", dropped_optional_groups if dropped_optional_groups else "none")
if len(FEATURE_COLUMNS) > 300:
    raise RuntimeError(f"Feature count {len(FEATURE_COLUMNS)} exceeds memory-safe cap of 300")
print(FEATURE_COLUMNS)

X_train = train_df[FEATURE_COLUMNS].replace([np.inf, -np.inf], np.nan).fillna(-999.0).astype("float32")
y_train = train_df["target_residual"].astype("float32")
X_test = test_df[FEATURE_COLUMNS].replace([np.inf, -np.inf], np.nan).fillna(-999.0).astype("float32")
log_memory("after matrix creation", X_train=X_train, X_test=X_test)


## Optional small well-level validation

In [ ]:
if RUN_LOCAL_VALIDATION:
    import lightgbm as lgb
    wells = np.array(sorted(train_df["well"].unique()))
    rng = np.random.default_rng(SEED)
    rng.shuffle(wells)
    val_n = max(1, int(len(wells) * VALIDATION_FRAC_WELLS))
    val_wells = set(wells[:val_n])
    val_mask = train_df["well"].isin(val_wells).to_numpy()
    local_params = dict(LGB_PARAMS)
    local_params["random_state"] = SEEDS[0]
    local_model = lgb.LGBMRegressor(**local_params)
    local_model.fit(X_train.loc[~val_mask], y_train.loc[~val_mask], callbacks=[lgb.log_evaluation(100)])
    val_pred = local_model.predict(X_train.loc[val_mask])
    print(f"small well-level residual RMSE: {rmse(y_train.loc[val_mask], val_pred):.6f}")
    del local_model, val_pred
    gc.collect()
else:
    print("RUN_LOCAL_VALIDATION=False; skipping local validation by default.")


## LightGBM training with GPU fallback

In [ ]:
try:
    import lightgbm as lgb
except Exception as exc:
    lgb = None
    print(f"LightGBM import failed; fallback model will be used: {type(exc).__name__}: {exc}")


class ZeroResidualModel:
    def fit(self, X, y):
        return self
    def predict(self, X):
        return np.zeros(len(X), dtype="float64")


def fit_lgbm_one_seed(X, y, seed: int):
    if lgb is None:
        raise RuntimeError("LightGBM is unavailable")
    base = dict(LGB_PARAMS)
    base["random_state"] = int(seed)
    gpu_params = dict(base)
    gpu_params.update(device_type="gpu", gpu_use_dp=False)
    try:
        print(f"Training LightGBM seed={seed} with GPU parameters...")
        model = lgb.LGBMRegressor(**gpu_params)
        model.fit(X, y, callbacks=[lgb.log_evaluation(100)])
        return model, "lgbm_gpu"
    except Exception as exc:
        print(f"GPU LightGBM failed for seed={seed}, retrying CPU LightGBM: {type(exc).__name__}: {exc}")
        gc.collect()
        cpu_params = dict(base)
        model = lgb.LGBMRegressor(**cpu_params)
        model.fit(X, y, callbacks=[lgb.log_evaluation(100)])
        return model, "lgbm_cpu"


models = []
model_records = []
for seed in SEEDS:
    try:
        model, used = fit_lgbm_one_seed(X_train, y_train, seed)
        models.append(model)
        model_records.append({"seed": seed, "model_used": used})
        log_memory(f"after model training seed {seed}", X_train=X_train)
    except Exception as exc:
        print(f"LightGBM seed={seed} failed completely: {type(exc).__name__}: {exc}")
        gc.collect()

if models:
    unique_modes = sorted({rec["model_used"] for rec in model_records})
    model_used = "lgbm_seed_ensemble_" + "+".join(unique_modes) if len(models) > 1 else unique_modes[0]
else:
    print("All LightGBM fits failed; using zero-residual anchor fallback so submission.csv is still generated.")
    models = [ZeroResidualModel().fit(X_train, y_train)]
    model_records = [{"seed": None, "model_used": "fallback_zero_residual_anchor"}]
    model_used = "fallback"

print("final model used:", model_used)
print("model records:", model_records)


## Residual prediction and submission save

In [ ]:
def residual_clip(delta, train_delta):
    qlo, qhi = np.nanquantile(np.asarray(train_delta, dtype="float64"), RESIDUAL_CLIP_QUANTILES)
    print("residual clipping bounds:", float(qlo), float(qhi))
    return np.clip(np.asarray(delta, dtype="float64"), qlo, qhi)


def confidence_gate(frame: pd.DataFrame) -> np.ndarray:
    n = len(frame)
    gate = np.full(n, 0.88, dtype="float64")
    corr = frame.get("known_typewell_gr_corr", pd.Series(0.0, index=frame.index)).to_numpy(dtype="float64")
    gr_miss = frame.get("gap_gr_nan_frac", pd.Series(0.35, index=frame.index)).to_numpy(dtype="float64")
    local_gap = frame.get("tw_local_top2_gap", pd.Series(0.0, index=frame.index)).to_numpy(dtype="float64")
    local_abs = frame.get("tw_local_min_absdiff", pd.Series(8.0, index=frame.index)).to_numpy(dtype="float64")
    path_gap = frame.get("off_path_top2_gap", pd.Series(0.0, index=frame.index)).to_numpy(dtype="float64")
    curvature = frame.get("trajectory_curvature", pd.Series(0.0, index=frame.index)).to_numpy(dtype="float64")
    frac = frame.get("off_tail_frac", pd.Series(0.5, index=frame.index)).to_numpy(dtype="float64")
    gate += 0.08 * np.clip(np.nan_to_num(corr, nan=0.0), -0.5, 1.0)
    gate -= 0.12 * np.clip((np.nan_to_num(gr_miss, nan=0.35) - 0.30) / 0.45, 0.0, 1.0)
    gate += 0.08 * np.clip(np.nan_to_num(local_gap, nan=0.0) / 8.0, 0.0, 1.0)
    gate += 0.05 * np.clip(np.nan_to_num(path_gap, nan=0.0) / 8.0, 0.0, 1.0)
    gate -= 0.10 * np.clip((np.nan_to_num(local_abs, nan=8.0) - 6.0) / 18.0, 0.0, 1.0)
    gate -= 0.05 * np.clip(np.abs(np.nan_to_num(curvature, nan=0.0)) / 0.2, 0.0, 1.0)
    gate += 0.04 * np.sin(np.pi * np.clip(frac, 0.0, 1.0))
    return np.clip(gate, 0.55, 1.0)


def path_prior_delta(frame: pd.DataFrame) -> np.ndarray:
    if "off_path_prior_delta" in frame.columns:
        prior = frame["off_path_prior_delta"].to_numpy(dtype="float64")
    else:
        frac = frame.get("off_tail_frac", pd.Series(0.0, index=frame.index)).to_numpy(dtype="float64")
        ease = 3.0 * np.clip(frac, 0.0, 1.0) ** 2 - 2.0 * np.clip(frac, 0.0, 1.0) ** 3
        endpoint = frame.get("off_path_soft_endpoint_mean", pd.Series(0.0, index=frame.index)).to_numpy(dtype="float64")
        prior = endpoint * ease
    return np.nan_to_num(prior, nan=0.0, posinf=0.0, neginf=0.0)


raw_pred_parts = []
for rec, model in zip(model_records, models):
    pred_part = np.asarray(model.predict(X_test), dtype="float64")
    raw_pred_parts.append(pred_part)
    print_stats(f"pred_residual raw seed {rec['seed']}", pred_part)

pred_residual = np.mean(np.vstack(raw_pred_parts), axis=0)
print_stats("pred_residual raw ensemble", pred_residual)

if USE_CURVE_CORRECTION:
    prior_delta = path_prior_delta(test_df)
    print_stats("curve/path prior residual", prior_delta)
    pred_residual = (1.0 - CURVE_CORRECTION_WEIGHT) * pred_residual + CURVE_CORRECTION_WEIGHT * prior_delta
    print("USE_CURVE_CORRECTION=True; CURVE_CORRECTION_WEIGHT=", CURVE_CORRECTION_WEIGHT)
else:
    print("USE_CURVE_CORRECTION=False")

if USE_RESIDUAL_CLIPPING:
    pred_residual = residual_clip(pred_residual, y_train)
    print("USE_RESIDUAL_CLIPPING=True")
else:
    print("USE_RESIDUAL_CLIPPING=False")

if USE_GATED_BLEND:
    gate = confidence_gate(test_df)
    print_stats("gated blend multiplier", gate)
    pred_residual = pred_residual * gate
    print("USE_GATED_BLEND=True")
else:
    print("USE_GATED_BLEND=False")

print_stats("pred_residual final", pred_residual)
pred_tvt = test_df["last_known_tvt_row"].to_numpy(dtype="float64") + pred_residual
print_stats("final_pred - last_known_tvt_row", pred_tvt - test_df["last_known_tvt_row"].to_numpy(dtype="float64"))
print_stats("final pred_tvt", pred_tvt)
log_memory("after prediction", X_test=X_test)

if not np.all(np.isfinite(pred_tvt)):
    raise RuntimeError("Final pred_tvt contains NaN or inf before submission alignment")

sample = pd.read_csv(DATA_DIR / "sample_submission.csv")
pred = pd.DataFrame({"id": test_df["id"].astype(str).values, SUBMISSION_TARGET: pred_tvt})
submission = sample[["id"]].copy()
submission["id"] = submission["id"].astype(str)
submission = submission.merge(pred, on="id", how="left", validate="one_to_one")

if submission[SUBMISSION_TARGET].isna().any():
    raise RuntimeError(f"Missing predictions for {int(submission[SUBMISSION_TARGET].isna().sum())} sample rows")
submission = submission[list(sample.columns)]
assert list(submission.columns) == list(sample.columns), "columns do not match sample_submission"
assert len(submission) == len(sample), "row count does not match sample_submission"
assert submission["id"].astype(str).tolist() == sample["id"].astype(str).tolist(), "id order does not match sample_submission"
assert SUBMISSION_TARGET in submission.columns, "tvt column missing"
assert np.all(np.isfinite(submission[SUBMISSION_TARGET].to_numpy(dtype="float64"))), "submission contains NaN or inf"

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(OUTPUT_PATH, index=False)
assert OUTPUT_PATH.exists(), "submission.csv was not written"
log_memory("after submission creation", submission=submission)
print(f"saved: {OUTPUT_PATH}")
print(submission.head())
print(submission[SUBMISSION_TARGET].describe())
